> 在线讲义：[第 10 章 · 策略评估 policy evaluation：从 0 开始，反复套 Bellman 方程](https://wikieden.github.io/rl-from-scratch/lessons/10-policy-eval.html) · 本书《从零学强化学习》

# 第 10 章 · 策略评估 policy evaluation：从 0 开始，反复套 Bellman 方程

本章回答：状态很多、方程组解不动时，怎么算出一份策略的价值？"反复代入"为什么会收敛到正确答案？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/10-policy-eval.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| 一轮更新（两张表） | `sweep(P, pi, v, gamma)` | 新的一整张表 |
| 第 0、1、2、3、10 轮 | `policy_evaluation(..., history=True)` 的 `hist[k]` | 图 4.1 左列 |
| 收敛 | 258 轮 | 0、−14、−20、−22 … |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**常量 `MOVES`**（来自 `rl_envs.py`）

In [ ]:
MOVES = [(-1, 0), (1, 0), (0, 1), (0, -1)]

**函数 `gridworld_4x4`**（来自 `rl_envs.py`）：S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。

In [ ]:
def gridworld_4x4():
    """S&B Example 4.1：4×4，左上 (0) 和右下 (15) 是终点，每走一步 −1，撞墙原地不动。"""
    n = 4
    P = {}
    for s in range(n * n):
        r, c = divmod(s, n)
        P[s] = {}
        for a, (dr, dc) in enumerate(MOVES):
            if s in (0, 15):
                P[s][a] = [(1.0, s, 0.0, True)]
                continue
            nr, nc = r + dr, c + dc
            ns = nr * n + nc if (0 <= nr < n and 0 <= nc < n) else s
            P[s][a] = [(1.0, ns, -1.0, ns in (0, 15))]
    return P

**函数 `q_from_v`**（来自 `ch08_bellman.py`）：q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。

In [ ]:
def q_from_v(P, v, gamma, s, a):
    """q_π(s, a) = Σ p(s', r|s, a) [r + γ v_π(s')]。"""
    return sum(p * (r + (0 if done else gamma * v[ns])) for p, ns, r, done in P[s][a])

**函数 `sweep`**（来自 `ch10_policy_eval.py`）：一轮：每个状态都用旧的 v 算一遍右边（同步更新，书里叫 two-array 版本）。

In [ ]:
def sweep(P, pi, v, gamma):
    """一轮：每个状态都用旧的 v 算一遍右边（同步更新，书里叫 two-array 版本）。"""
    return np.array([sum(pa * q_from_v(P, v, gamma, s, a) for a, pa in pi[s].items()) for s in P])

**函数 `policy_evaluation`**（来自 `ch10_policy_eval.py`）：下面要用到的小工具。

In [ ]:
def policy_evaluation(P, pi, gamma=1.0, theta=1e-6, history=False):
    v, hist = np.zeros(len(P)), []
    while True:
        hist.append(v)
        new = sweep(P, pi, v, gamma)
        if np.max(np.abs(new - v)) < theta:
            return (new, hist) if history else new
        v = new

**函数 `uniform_policy`**（来自 `rl_envs.py`）：下面要用到的小工具。

In [ ]:
def uniform_policy(P):
    return {s: {a: 1.0 / len(P[s]) for a in P[s]} for s in P}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
P = gridworld_4x4()

In [ ]:
v, hist = policy_evaluation(P, uniform_policy(P), history=True)

In [ ]:
for k in [0, 1, 2, 3, 10]:
        print(f"k = {k}：\n{np.round(hist[k].reshape(4, 4), 1)}")

In [ ]:
print(f"收敛（{len(hist)} 轮）：\n{np.round(v.reshape(4, 4), 1)}")

## 3. 代码里才看得到的

终点在代码里写成"做什么都留在原地、奖励 0、`done=True`"，`q_from_v` 遇到 `done` 就不加 $\gamma v(s')$，所以终点的价值一直是 0。把 `sweep` 改成"算一格马上写回 v"就是原地版本，可以自己试试它要几轮收敛。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. `policy_evaluation(..., theta=1e-2, history=True)`：88 轮就停，价值是 0、−13.9、−19.84、−21.83；`theta=1e-6` 要 257 轮，才到 −14、−20、−22。θ 决定"差多少算收敛"。
2. 把 γ 从 1 改成 0.9：88 轮收敛，第一行变成 0、−5.28、−7.13、−7.65。打折以后"远处的 −1"变轻了，所以数小得多。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 方程当规则：旧值代右边，新值写左边；
> 一轮一轮套，终点的 0 往外渗；
> 误差每轮缩，最后停在真价值；
> 两张表稳当，一张表更快。